# 14_train_evaluate_supervised_ML_models

Run supervised ML training/evaluation across feature combinations, split types, and model choices.

## 1) Environment Setup


In [1]:
from pathlib import Path
import sys

# Section 1: Resolve repo root from either project root or notebooks/ cwd.
cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd

# Section 2: Add repo and src roots to sys.path for imports.
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

print("repo_root:", repo_root)
print("src_root:", src_root)


repo_root: /Users/charmainechia/Documents/projects/agentic-protein-design
src_root: /Users/charmainechia/Documents/projects/agentic-protein-design/src


## 2) Imports


In [2]:
from pprint import pprint
from project_config.feature_registry import COMBI_ML_FEATURE_SETS
from project_config.variables import address_dict


## 3) Optional Data Segmentation


In [3]:
from tools.ml.segment_data import run_data_segmentation

# Optional: run segmentation before ML.
segmentation_inputs = {
    'run_segmentation': False,
    'root_key': 'MUTAGENESIS-DATA-BENCHMARKS',
    'data_subfolder': 'D7PM05_CLYGR_Somermeyer_2022',
    'csv_file': 'D7PM05_CLYGR_Somermeyer_2022.csv',
    'output_csv_file': 'D7PM05_CLYGR_Somermeyer_2022_0.csv',  # optional override
    'k_folds': 5,
    'mutation_separator': ':',
    'num_mutation_segments_singlemut': 5,
    'min_layer_size_for_multimut_segmentation': 1000,
    'max_layer_size_for_multimut_segmentation': [None,1000],
    'smallest_single_mutant_size': 70,
    'include_mutation_onehot_for_clustering': True,
    'verbose': True,
}

segmentation_result = {'status': 'skipped', 'output_dataset_fname': None}
if segmentation_inputs['run_segmentation']:
    csv_dir = (repo_root / Path(address_dict[segmentation_inputs['root_key']]) / 'expdata' / segmentation_inputs['data_subfolder']).resolve()
    payload = dict(segmentation_inputs)
    payload['csv_dir'] = str(csv_dir)
    segmentation_result = run_data_segmentation(payload)

print(segmentation_result)


Loading CSV from: /Users/charmainechia/Documents/projects/MUTAGENESIS-DATA-BENCHMARKS/expdata/D7PM05_CLYGR_Somermeyer_2022/D7PM05_CLYGR_Somermeyer_2022.csv

=== Dataset Stats ===
Total mutants: 24515
Unique mutants: 24515
Unique individual mutation tokens: 1651
Unique mutated positions: 234
Percent of mutants by num_mutations:
  num_mutations=1: 1169 (4.77%)
  num_mutations=2: 10148 (41.40%)
  num_mutations=3: 6315 (25.76%)
  num_mutations=4: 3417 (13.94%)
  num_mutations=5: 1752 (7.15%)
  num_mutations=6: 935 (3.81%)
  num_mutations=7: 427 (1.74%)
  num_mutations=8: 220 (0.90%)
  num_mutations=9: 76 (0.31%)
  num_mutations=10: 33 (0.13%)
  num_mutations=11: 16 (0.07%)
  num_mutations=12: 3 (0.01%)
  num_mutations=13: 2 (0.01%)
  num_mutations=14: 1 (0.00%)
  num_mutations=23: 1 (0.00%)

=== Single-Mutation Segmentation (Primary) ===
Unique positions among single mutants: 234
Configured segments: 5
  segment_0: 248 mutants
  segment_1: 229 mutants
  segment_2: 230 mutants
  segment_3: 

## 4) Configure ML Inputs


In [4]:
import agentic_protein_design.steps.train_evaluate_supervised_ml_models as ml_step
default_user_inputs = ml_step.default_user_inputs
user_inputs = default_user_inputs()

# Edit these for your run
user_inputs['root_key'] = 'MUTAGENESIS-DATA-BENCHMARKS' # 'examples' # 'ECOHARVEST' #
user_inputs['data_fbase'] = user_inputs['root_key']
user_inputs['data_subfolder'] = 'D7PM05_CLYGR_Somermeyer_2022' # 'ET096_R1-2' # 'RML_R1' #
user_inputs['csv_suffix'] = '_0' # '' # '_3'
default_dataset_fname = user_inputs['data_subfolder'] + user_inputs['csv_suffix'] + '.csv'
user_inputs['dataset_fname'] = segmentation_result.get('output_dataset_fname') or default_dataset_fname

# Experimental data parent folder: address_dict[root_key] / "expdata" / data_subfolder
expdata_dir = (repo_root / Path(address_dict[user_inputs['root_key']]) / 'expdata' / user_inputs['data_subfolder']).resolve()
user_inputs['input_filename_prefix'] = user_inputs['data_subfolder'] + '_' # ''
user_inputs['sequence_base'] = 'sequences/D7PM05_CLYGR_Somermeyer_2022.fasta' # "sequences/ET096.fasta"
user_inputs['target_col'] = ['DMS_score'] # [f'foldchange_{s}_activity_25C' for s in ['NBD','ABTS']] # ['foldchange_TSO'] #
user_inputs['classification_or_regression'] = 'regression'

# specify splits
user_inputs['split_type_list'] = ['custom', 'mutres-modulo', 'random'] # ['mutres-modulo', 'random'] # ['mutres-modulo', 'custom'] # ['random', 'custom'] #
user_inputs['k_folds'] = 5
user_inputs['random_kfold_repeats'] = 1  # >1 enables repeated random k-fold from scratch
user_inputs['random_split_col'] = f'fold_random_{user_inputs["k_folds"]}'
user_inputs['mutres_split_col'] = f'fold_mutres-modulo_{user_inputs["k_folds"]}'
user_inputs['contiguous_split_col'] = f'fold_contiguous_{user_inputs["k_folds"]}'  # optional precomputed column
user_inputs['custom_split_col'] = 'fold_custom'
user_inputs['custom_test_value'] = 1
user_inputs['segment_col'] = 'segment_index_1'  # which segmentation column to use

# specify custom dataset name
user_inputs['custom_test_dataset_fname'] = None # 'ET096_R3.csv' # ''
user_inputs['custom_test_data_subfolder'] = user_inputs['data_subfolder']
user_inputs['custom_input_filename_prefix'] = None if user_inputs['custom_test_dataset_fname'] is None else user_inputs['custom_test_dataset_fname'].split('.')[0] # ''

# specify featureset combinations to test
# user_inputs['feature_combinations_dict'] = COMBI_ML_FEATURE_SETS
user_inputs['feature_combinations_dict'] = {
    'onehot_esm2_LLR': ['one_hot', 'esm2-650m_LLR-masked'],
    'georgiev_esm2_LLR': ['georgiev', 'esm2-650m_LLR-masked'],
    'esm2_seq_embeddings_LLR': ['esm2-650m_mean_pooled-33', 'esm2-650m_LLR-masked'],
}

# specify models
user_inputs['model_list'] = ['ridge', 'xgboost'] # ['mlp_sklearn', 'mlp_pytorch'] #

# run parameters
# hyperparameter tuning (applied from first fold to subsequent folds in each segment)
user_inputs['run_hyperparameter_tuning'] = False
user_inputs['tuning_metric'] = 'spearman'
user_inputs['tuning_n_trials'] = 20

# save parameters
user_inputs['save_trained_models'] = False
user_inputs['save_predictions'] = False
user_inputs['train_full_data_model'] = False
user_inputs['featurecombi_model_pair_to_extract_coefficients_for'] = None # [('onehot', 'ridge')]
user_inputs['show_progress'] = True
pprint(user_inputs)


{'classification_or_regression': 'regression',
 'contiguous_split_col': 'fold_contiguous_5',
 'csv_suffix': '_0',
 'custom_input_filename_prefix': None,
 'custom_split_col': 'fold_custom',
 'custom_test_data_subfolder': 'D7PM05_CLYGR_Somermeyer_2022',
 'custom_test_dataset_fname': None,
 'custom_test_value': 1,
 'data_fbase': 'MUTAGENESIS-DATA-BENCHMARKS',
 'data_subfolder': 'D7PM05_CLYGR_Somermeyer_2022',
 'dataset_fname': 'D7PM05_CLYGR_Somermeyer_2022_0.csv',
 'feature_combinations_dict': {'esm2_seq_embeddings_LLR': ['esm2-650m_mean_pooled-33',
                                                           'esm2-650m_LLR-masked'],
                               'georgiev_esm2_LLR': ['georgiev',
                                                     'esm2-650m_LLR-masked'],
                               'onehot_esm2_LLR': ['one_hot',
                                                   'esm2-650m_LLR-masked']},
 'featurecombi_model_pair_to_extract_coefficients_for': None,
 'input_filename_pr

## 5) Run Training And Evaluation


In [ ]:
import importlib

import tools.ml.model_registry as model_registry
import tools.ml.hyperparameter_tuning as hyperparameter_tuning
import tools.ml.workflow as workflow
import agentic_protein_design.steps.train_evaluate_supervised_ml_models as ml_step

importlib.reload(model_registry)
importlib.reload(hyperparameter_tuning)
importlib.reload(workflow)
ml_step = importlib.reload(ml_step)

train_evaluate_supervised_ml_models = ml_step.train_evaluate_supervised_ml_models

result = train_evaluate_supervised_ml_models(user_inputs)
pprint(result)


[eval-start] target_col=DMS_score | split_type=custom | feature_combi_name=onehot_esm2_LLR | model_name=ridge
[fold-result] split_id=0 | split_name=custom_segment_1 | split_type=custom | eval_group=segments_0_to_1 | data_size_n=150 | segments_included=[0,1] | n_train=75 | n_test=75
 test_spearman  test_pearson  test_r2  test_rmse  train_spearman  train_pearson  train_r2  train_rmse
        0.1536        0.0305  -0.0068 10987.2599          0.9312         0.9714    0.8874   3578.3304
[eval-start] target_col=DMS_score | split_type=custom | feature_combi_name=onehot_esm2_LLR | model_name=ridge
[fold-result] split_id=1 | split_name=custom_segment_2 | split_type=custom | eval_group=segments_0_to_2 | data_size_n=229 | segments_included=[0,1,2] | n_train=150 | n_test=79
 test_spearman  test_pearson  test_r2  test_rmse  train_spearman  train_pearson  train_r2  train_rmse
       -0.2951       -0.2965  -0.0546 12053.9777          0.9365         0.9728    0.8914   3561.4809
[eval-start] target_col

## 6) Visualize ML Results


In [ ]:
from pprint import pprint

import agentic_protein_design.steps.visualize_ml_results as ml_viz_step

visualize_ml_results = ml_viz_step.visualize_ml_results
default_viz_inputs = ml_viz_step.default_user_inputs

viz_inputs = default_viz_inputs()
viz_inputs['data_fbase'] = user_inputs.get('data_fbase', 'examples')
viz_inputs['data_subfolder'] = user_inputs.get('data_subfolder', '')
viz_inputs['metrics_summary_fname'] = f"{user_inputs['classification_or_regression']}_metrics_summary{user_inputs.get('csv_suffix', '')}.csv"
viz_inputs['model_name_list'] = ['best']  # e.g. ['xgboost', 'ridge', 'best']
viz_inputs['metric_col'] = ['test_spearman']  # string or list
viz_inputs['split_type_list'] = []  # [] -> iterate all split types in summary CSV
viz_inputs['feature_label_list'] = list(user_inputs['feature_combinations_dict'].keys())
viz_inputs['save_figure'] = True
viz_inputs['show_figure'] = True
viz_inputs['figure_output_dir'] = ''  # default: summary csv folder
viz_inputs['figure_fname'] = ''       # default auto name
viz_inputs['y_limits'] = [0, 1]      # e.g. [0.0, 1.0]; None/[] uses matplotlib defaults

# Optional override:
# viz_inputs['metrics_summary_csv_path'] = '/Users/charmainechia/Documents/projects/MUTAGENESIS-DATA-BENCHMARKS/ml/output/Q65J43_BACLD_g4_Thomas_2025/regression_metrics_summary_2.csv'

viz_result = visualize_ml_results(viz_inputs)
pprint(viz_result)
